# Comparing multiple CaImAn CNMF pipelines (extended version)

This notebook loads several `.hdf5` CNMF outputs that were run with different parameters
(for example different `gSig`, merge thresholds, or component selection rules) and helps you
compare which parameter set you can trust more.

For each **model** (one `.hdf5` CNMF result) it will:

* load the CNMF object using `caiman.source_extraction.cnmf.cnmf.load_CNMF`
* make sure quality metrics exist (run `evaluate_components` if needed and possible)
* extract and analyse **per‑component metrics**:

  - spatial SNR (`snr_comp`, or equivalent)
  - spatial correlation (`r_values`)
  - CNN prediction score (`cnn_preds`)
  - accepted vs rejected label (`idx_components`)
  - **additional metrics**:
      * footprint area (number of pixels in `A`)
      * event count (from deconvolved spikes if available)
      * temporal trace SNR
      * max neighbour correlation (temporal correlation with the most similar component)

* create summary tables and plots:
  - r vs SNR and CNN vs SNR scatter plots, per model
  - histograms of SNR / r / CNN distributions across models
  - histograms of the new metrics across models
  - a summary table with medians per model

You can then use `nb_view_components` to visually inspect components for each model.
Set the paths in the configuration cell below to your own `.hdf5` files.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from caiman.source_extraction.cnmf.cnmf import load_CNMF


plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

## 1. Configure your models

Edit `MODEL_PATHS` so that each key is a short name for a pipeline
and each value is the full path to the corresponding CNMF `.hdf5` file.

Set the thresholds to the values you normally use.
These are used only for plotting and summaries, **they do not change the CNMF results**.


In [ ]:
MODEL_PATHS = {
    "final": "/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_final/caiman_results.hdf5",
    "biblio": "/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_biblio/caiman_results.hdf5",
    "biblio_contoursize": "/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_biblio_contoursize/caiman_results.hdf5",
}



RVAL_THR = 0.8
MIN_SNR = 2.5
MIN_CNN = 0.9


# Optional: recording info (used only to convert event counts to rates)
FRAME_RATE = 25.0  # Hz, set to your recording frame rate


## 2. Helper functions: loading models and extracting metrics

In [ ]:
def load_models(model_paths):
    """Load several CNMF results from .hdf5 files.

    Returns
    -------
    dict
        {model_name: cnm} mapping, skipping files that do not exist.
    """
    models = {}
    for name, path in model_paths.items():
        if not os.path.exists(path):
            print(f"Warning: file for '{name}' not found: {path}")
            continue
        print(f"Loading {name} from {path}")
        models[name] = load_CNMF(path)
    return models

In [ ]:
def _get_metric(est, candidates, metric_name, model_name):
    """Try several possible attribute names and return as np.array.

    If nothing is found, raise a clear AttributeError.
    """
    for attr in candidates:
        if hasattr(est, attr):
            val = getattr(est, attr)
            if val is not None:
                return np.array(val)
    raise AttributeError(
        f"{metric_name} not found in Estimates for model '{model_name}'. "
        "You may need to run evaluate_components manually."
    )


def ensure_quality_metrics(cnm, model_name=""):
    """Make sure snr / r / cnn metrics exist on cnm.estimates.

    If not, try to recompute them with evaluate_components using the saved memmap.
    If this fails, the user will need to run evaluate_components manually.
    """
    est = cnm.estimates

    # Heuristic: if any of these exist, assume evaluate_components has already run
    if hasattr(est, "snr_comp") or hasattr(est, "SNR_comp") or hasattr(est, "r_values") or hasattr(est, "cnn_preds") or hasattr(est, "cnn_preds_spatial"):
        return

    print(f"Metrics missing for model '{model_name}', trying to run evaluate_components...")

    if cnm.mmap_file is None:
        print(
            f"  -> cnm.mmap_file is None for '{model_name}'. "
            "Load your movie as Y and call est.evaluate_components(Y, cnm.params, dview=None) manually."
        )
        return

    try:
        Yr, dims, T = load_memmap(cnm.mmap_file)
        Y = Yr.T.reshape((T,) + dims, order="F")
        cnm.estimates.evaluate_components(Y, cnm.params, dview=None)
        print("  -> evaluate_components finished.")
    except Exception as e:
        print(f"  -> Could not run evaluate_components for '{model_name}': {e}")


def extract_metrics(cnm, model_name=""):
    """Extract quality metrics and accepted / rejected labels from a CNMF object.

    Returns a dict of 1D arrays with keys:
      snr, r, cnn, accepted, area, event_count, trace_snr, max_neigh_corr
    """
    est = cnm.estimates

    # Ensure basic metrics exist
    ensure_quality_metrics(cnm, model_name=model_name)

    # Basic CaImAn quality metrics
    snr = _get_metric(est, ["snr_comp", "SNR_comp"], "SNR", model_name)
    r   = _get_metric(est, ["r_values"], "r_values", model_name)
    cnn = _get_metric(est, ["cnn_preds", "cnn_preds_spatial"], "cnn_preds", model_name)

    n_comp = len(snr)

    # Accepted / rejected components
    accepted = np.zeros(n_comp, dtype=bool)
    if hasattr(est, "idx_components") and est.idx_components is not None:
        accepted[est.idx_components] = True
    else:
        print(f"Warning: idx_components not found for '{model_name}', assuming all rejected.")

    # Spatial footprints: area per component
    A = est.A  # usually a sparse matrix (pixels × components)
    try:
        # This is efficient for scipy.sparse matrices
        area = A.getnnz(axis=0)
    except Exception:
        A_arr = A.toarray() if hasattr(A, "toarray") else np.array(A)
        area = np.sum(A_arr > 0, axis=0)
    area = np.asarray(area).ravel()

    # Temporal traces
    C = est.C  # shape (K × T)
    if C is None:
        C = np.zeros((n_comp, 1))

    # Temporal SNR (trace_snr)
    #   difference between 95th and 5th percentiles divided by baseline std
    baseline_std = np.std(C, axis=1) + 1e-8
    amp_range = np.percentile(C, 95, axis=1) - np.percentile(C, 5, axis=1)
    trace_snr = amp_range / baseline_std

    # Event count from deconvolved spikes if available
    event_trains = None
    for attr in ["S", "spikes", "S_", "D"]:
        if hasattr(est, attr) and getattr(est, attr) is not None:
            event_trains = getattr(est, attr)
            break
    if event_trains is not None:
        event_trains = np.array(event_trains)
        if event_trains.ndim == 1:
            event_trains = event_trains[None, :]
        event_count = np.sum(event_trains > 0, axis=1)
    else:
        event_count = np.full(n_comp, np.nan)

    # Max neighbour correlation between traces
    if C.shape[1] > 1:
        corr_mat = np.corrcoef(C)
        # Remove self-correlation by setting diagonal to 0
        corr_mat = corr_mat - np.eye(corr_mat.shape[0])
        max_neigh_corr = np.nanmax(corr_mat, axis=1)
    else:
        max_neigh_corr = np.zeros(n_comp)

    metrics = dict(
        snr=snr,
        r=r,
        cnn=cnn,
        accepted=accepted,
        area=area,
        event_count=event_count,
        trace_snr=trace_snr,
        max_neigh_corr=max_neigh_corr,
    )
    return metrics


def build_long_dataframe(models):
    """For a dict {name: cnm} return a long-form DataFrame with one row per component.

    Columns: model, comp_idx, snr, r, cnn, accepted, area, event_count, trace_snr, max_neigh_corr
    """
    rows = []
    for name, cnm in models.items():
        m = extract_metrics(cnm, model_name=name)
        n = len(m["snr"])
        for i in range(n):
            rows.append(
                dict(
                    model=name,
                    comp_idx=i,
                    snr=m["snr"][i],
                    r=m["r"][i],
                    cnn=m["cnn"][i],
                    accepted=bool(m["accepted"][i]),
                    area=m["area"][i],
                    event_count=m["event_count"][i],
                    trace_snr=m["trace_snr"][i],
                    max_neigh_corr=m["max_neigh_corr"][i],
                )
            )
    return pd.DataFrame(rows)

In [ ]:
def summarize_model(df):
    """Return a summary DataFrame grouped by model."""
    summaries = []
    for name, sub in df.groupby("model"):
        acc = sub[sub["accepted"]]
        summaries.append(
            dict(
                model=name,
                n_total=len(sub),
                n_accepted=len(acc),
                frac_accepted=len(acc) / len(sub) if len(sub) > 0 else np.nan,
                median_snr_acc=np.median(acc["snr"]) if len(acc) > 0 else np.nan,
                median_r_acc=np.median(acc["r"]) if len(acc) > 0 else np.nan,
                median_cnn_acc=np.median(acc["cnn"]) if len(acc) > 0 else np.nan,
                median_area_acc=np.median(acc["area"]) if len(acc) > 0 else np.nan,
                median_event_count_acc=np.median(acc["event_count"]) if len(acc) > 0 else np.nan,
                median_trace_snr_acc=np.median(acc["trace_snr"]) if len(acc) > 0 else np.nan,
                median_max_neigh_corr_acc=np.median(acc["max_neigh_corr"]) if len(acc) > 0 else np.nan,
            )
        )
    return pd.DataFrame(summaries).set_index("model")

## 3. Load all models and build the combined DataFrame

In [ ]:
cnm_models = load_models(MODEL_PATHS)

df_all = build_long_dataframe(cnm_models)
print("First few rows:")
display(df_all.head())

summary = summarize_model(df_all)
print("\nSummary per model (accepted components only):")
display(summary)

## 4. r vs SNR and CNN vs SNR per model

Grey = rejected components; green = accepted components.  
Vertical and horizontal dashed lines show your chosen thresholds.


In [ ]:
def plot_r_vs_snr(df_model, title):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    # rejected
    rej = df_model[~df_model["accepted"]]
    ax.scatter(rej["r"], rej["snr"], s=10, alpha=0.3, color="gray", label="Rejected")

    # accepted
    acc = df_model[df_model["accepted"]]
    ax.scatter(acc["r"], acc["snr"], s=20, alpha=0.9, color="green", label="Accepted")

    ax.axvline(RVAL_THR, linestyle="--", color="red", label=f"rval_thr={RVAL_THR}")
    ax.axhline(MIN_SNR, linestyle="--", color="orange", label=f"min_SNR={MIN_SNR}")

    ax.set_xlabel("Spatial corr (r)")
    ax.set_ylabel("SNR")
    ax.set_title(title)
    ax.legend()
    plt.show()


def plot_cnn_vs_snr(df_model, title):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    # rejected
    rej = df_model[~df_model["accepted"]]
    ax.scatter(rej["cnn"], rej["snr"], s=10, alpha=0.3, color="gray", label="Rejected")

    # accepted
    acc = df_model[df_model["accepted"]]
    ax.scatter(acc["cnn"], acc["snr"], s=20, alpha=0.9, color="green", label="Accepted")

    ax.axvline(MIN_CNN, linestyle="--", color="red", label=f"min_cnn_thr={MIN_CNN}")
    ax.axhline(MIN_SNR, linestyle="--", color="orange", label=f"min_SNR={MIN_SNR}")

    ax.set_xlabel("CNN prediction score")
    ax.set_ylabel("SNR")
    ax.set_title(title)
    ax.legend()
    plt.show()

In [ ]:
for name, df_m in df_all.groupby("model"):
    print(f"Model: {name}")
    plot_r_vs_snr(df_m, f"{name}: r vs SNR")
    plot_cnn_vs_snr(df_m, f"{name}: CNN vs SNR")

## 5. Compare metric distributions across models

We now compare distributions of both classical and new metrics:
SNR, spatial correlation, CNN score, temporal trace SNR, footprint area,
event count, and max neighbour correlation.


In [ ]:
def plot_hist_comparisons(df_all, n_bins=40):
    models = sorted(df_all["model"].unique())
    metrics = [
        ("snr", "SNR"),
        ("r", "Spatial corr (r)"),
        ("cnn", "CNN"),
        ("trace_snr", "Trace SNR"),
        ("area", "Footprint area (pixels)"),
        ("event_count", "Event count"),
        ("max_neigh_corr", "Max neighbour corr"),
    ]

    n_cols = 3
    n_rows = int(np.ceil(len(metrics) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
    axes = axes.flatten()

    for idx, (key, label) in enumerate(metrics):
        ax = axes[idx]
        for m in models:
            vals = df_all.loc[df_all["model"] == m, key].values
            vals = vals[~np.isnan(vals)]
            ax.hist(vals, bins=n_bins, histtype="stepfilled", alpha=0.4, label=m)
        ax.set_xlabel(label)
        ax.set_ylabel("Count")
        ax.set_title(label)

    # Remove unused axes if any
    for j in range(len(metrics), len(axes)):
        fig.delaxes(axes[j])

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(0.98, 0.98))
    plt.tight_layout()
    plt.show()


plot_hist_comparisons(df_all)

## 6. Visual inspection with `nb_view_components`

Numbers are useful, but visual inspection is essential.
Use the correlation image or average image as `cn_img`, then view a subset
of accepted components for each model.

The cell below uses the `Cn` image from the **first** model as background.
Change `cn_img` if you want a different image.


In [ ]:
# Use Cn (correlation image) from the first loaded model as default background
if len(cnm_models) > 0:
    first_key = list(cnm_models.keys())[0]
    cn_img = cnm_models[first_key].estimates.Cn
else:
    cn_img = None
    print("No models loaded, cannot define cn_img yet.")


def preview_components_all_models(cnm_models, img, step=5):
    """Call nb_view_components for each model on a subset of accepted components.

    Parameters
    ----------
    cnm_models : dict
        {name: cnm} mapping.
    img : 2D array
        Background image (correlation or mean image).
    step : int
        Show every `step`-th accepted component to keep it lightweight.
    """
    if img is None:
        print("cn_img is None, please set a background image first.")
        return

    for name, cnm in cnm_models.items():
        est = cnm.estimates
        if not hasattr(est, "idx_components") or est.idx_components is None or len(est.idx_components) == 0:
            print(f"Model '{name}' has no accepted components (idx_components empty). Skipping.")
            continue
        print(f"### Viewing components from model: {name}")
        idx = est.idx_components[::step]
        cnm.estimates.nb_view_components(
            img=img,
            idx=idx,
            cmap="gray",
            denoised_color="red",
        )


# Uncomment this line when you are ready to visually inspect components
# preview_components_all_models(cnm_models, cn_img, step=5)

To inspect a single model in more detail, use:


In [ ]:
# Example: preview components from a single model
model_to_look_at = "contour_size"  # change to one of the keys in MODEL_PATHS

if model_to_look_at in cnm_models and cn_img is not None:
    cnm = cnm_models[model_to_look_at]
    cnm.estimates.nb_view_components(
        img=cn_img,
        idx=cnm.estimates.idx_components[::5],
        cmap="gray",
        denoised_color="red",
    )
else:
    print("Model not found or cn_img is None. Check MODEL_PATHS and run previous cells.")

## 7. How to use all this to choose parameters

1. **Look at the summary table**  
   - `n_accepted` and `frac_accepted` tell you how strict each pipeline is.  
   - `median_snr_acc`, `median_r_acc`, `median_cnn_acc` show the classical CaImAn metrics.  
   - New medians (`median_trace_snr_acc`, `median_area_acc`, `median_event_count_acc`, `median_max_neigh_corr_acc`) tell you about temporal quality, footprint size, activity level, and redundancy.

2. **Inspect the scatter plots (r vs SNR, CNN vs SNR)**  
   - Accepted components should cluster in the high‑SNR and high‑r / high‑CNN region.  
   - If many accepted components lie close to thresholds and overlap with rejected ones, that pipeline may be too permissive.

3. **Compare histograms across models**  
   - Prefer models where the distributions of SNR, r, CNN, and trace SNR are shifted to higher values without collapsing the number of components.  
   - Footprint area histograms should roughly match expected soma size; avoid models that produce very large or extremely tiny footprints.  
   - Event count distributions: you usually want a reasonable fraction of cells with non‑zero event counts, not almost all silent components.  
   - Max neighbour correlation: extremely high values for many cells may indicate duplicates or global artefacts.

4. **Use `nb_view_components` for final judgement**  
   - Confirm that components look like real neurons (compact, plausible footprints, clean transients) rather than neuropil or artefact.  
   - Compare a few example cells from each model side‑by‑side; pick the parameter set that looks best empirically while keeping enough cells for your analysis.

In practice you will probably end up choosing one or two parameter sets that give a good trade‑off between **quality** (clean metrics, good‑looking cells) and **quantity** (enough accepted neurons).